In [0]:
df =  (

spark.read
.option("header", "true")
.option("inferSchema", "true")
.csv("/Volumes/workspace/default/dados/dataset_olx_raw.csv")
)



##BRONZE

In [0]:
df.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("workspace.default.bronze_olx")


#silver

In [0]:
from pyspark.sql import functions as F

#leitura_da_bronze
df_bronze = spark.table("workspace.default.bronze_olx")

#tratamento_camada_bronze

df_silver =(

    df_bronze

    #remover espaços extras
    .withColumn("titulo", F.trim(F.col("titulo")))
     .withColumn("url", F.trim(F.col("url")))
     .withColumn("tipo", F.trim(F.col("tipo")))
    .withColumn("bairro", F.trim(F.col("bairro")))
    .withColumn("cidade", F.trim(F.col("cidade")))
    .withColumn("estado", F.trim(F.col("estado")))
    .withColumn("cep", F.trim(F.col("cep")))
    .withColumn("descricao", F.trim(F.col("descricao")))

    #TRANSFORMAR TEXTOS VAZIOS EM NULL
    .replace("", None)


    #converter as colunas numericas
    .withColumn("preco", F.col("preco").cast("double"))
    .withColumn("quartos", F.col("quartos").cast("double"))
    .withColumn("banheiros", F.col("banheiros").cast("double"))
    .withColumn("garagens", F.col("garagens").cast("double"))
    .withColumn("area_m2", F.col("area_m2").cast("double"))

    #remover somente registros sem informação
    .na.drop(subset = ["titulo", "url"])

    #remove somente registros sem informação
    .dropDuplicates(["url"])

     )
                                

(

   df_silver.write\
    .mode("overwrite")\
    .saveAsTable("workspace.default.silver_olx")
   
)


#Gold

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, coalesce, lit, round, current_timestamp

df_silver = spark.table("workspace.default.silver_olx")
df_gold = (df_silver
.withColumn("preco_m2",
            when((col("area_m2").isNotNull()) & (col("area_m2") > 0),
                 round(col("preco") / col("area_m2"), 2))
)
.withColumn("categoria_preco",
            when(col("preco") < 300000, "baixo")
            .when(col("preco") < 700000, "medio")
            .otherwise("alto")
)
.withColumn("total_comodos",
            coalesce(col("quartos"), lit(0))
            + coalesce(col("banheiros"), lit(0))
)
.withColumn("Possui_garagem",
            when(col("garagens") > 0, "sim").otherwise("nao")
)
.withColumn("dt_carga",
            current_timestamp()
)
)

(

df_gold.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("workspace.default.gold_olx")
    )